# **Labs 1 PySpark:**

In these labs we will be using the "[[NeurIPS 2020] Data Science for COVID-19 (DS4C)](https://www.kaggle.com/datasets/kimjihoo/coronavirusdataset?select=PatientInfo.csv)" dataset, retrieved from [Kaggle](https://www.kaggle.com/) on 1/6/2022, for educational non commercial purpose, License
[CC BY-NC-SA 4.0
](https://creativecommons.org/licenses/by-nc-sa/4.0/)


The csv file that we will be using in this lab is **PatientInfo**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## PatientInfo.csv

**patient_id**
the ID of the patient

**sex**
the sex of the patient

**age**
the age of the patient

**country**
the country of the patient

**province**
the province of the patient

**city**
the city of the patient

**infection_case**
the case of infection

**infected_by**
the ID of who infected the patient


**contact_number**
the number of contacts with people

**symptom_onset_date**
the date of symptom onset

**confirmed_date**
the date of being confirmed

**released_date**
the date of being released

**deceased_date**
the date of being deceased

**state**
isolated / released / deceased

### Import the pyspark and check it's version

In [1]:
import pyspark
pyspark.__version__

'4.0.2'

### Import and create SparkSession

In [2]:
!pip install findspark

In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

### Load the PatientInfo.csv file and show the first 5 rows

In [4]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [5]:
df = spark.read.csv("/content/drive/MyDrive/Data/PatientInfo.csv", header=True, inferSchema=True)

In [6]:
df.show(5)

+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|       city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul| Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|         NULL|released|
|1000000002|  male|30s|  Korea|   Seoul|Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|         NULL|released|
|1000000003|  male|50s|  Korea|   Seoul|  Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|   202

### Display the schema of the dataset

In [7]:
df.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: string (nullable = true)
 |-- country: string (nullable = true)
 |-- province: string (nullable = true)
 |-- city: string (nullable = true)
 |-- infection_case: string (nullable = true)
 |-- infected_by: string (nullable = true)
 |-- contact_number: string (nullable = true)
 |-- symptom_onset_date: string (nullable = true)
 |-- confirmed_date: date (nullable = true)
 |-- released_date: date (nullable = true)
 |-- deceased_date: date (nullable = true)
 |-- state: string (nullable = true)



### Display the statistical summary

In [8]:
print(df.describe().show())
print("---------------------------")
print(df.summary().show())

+-------+--------------------+------+----+----------+--------+--------------+--------------------+--------------------+--------------------+------------------+--------+
|summary|          patient_id|   sex| age|   country|province|          city|      infection_case|         infected_by|      contact_number|symptom_onset_date|   state|
+-------+--------------------+------+----+----------+--------+--------------+--------------------+--------------------+--------------------+------------------+--------+
|  count|                5165|  4043|3785|      5165|    5165|          5071|                4246|                1346|                 791|               690|    5165|
|   mean|2.8636345618679576E9|  NULL|NULL|      NULL|    NULL|          NULL|                NULL|2.2845944015643125E9|1.6772572523506988E7|              NULL|    NULL|
| stddev| 2.074210725277473E9|  NULL|NULL|      NULL|    NULL|          NULL|                NULL|1.5265072953383324E9| 3.093097580985502E8|              N

### Using the state column.
### How many people survived (released), and how many didn't survive (isolated/deceased)?

In [9]:
df.groupBy("state").count().show()

+--------+-----+
|   state|count|
+--------+-----+
|isolated| 2158|
|released| 2929|
|deceased|   78|
+--------+-----+



### Display the number of null values in each column

In [10]:
df.select([c for c in df.columns if df.filter(df[c].isNull()).count() > 0]).show()

+------+---+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+
|   sex|age|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|
+------+---+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+
|  male|50s|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|         NULL|
|  male|30s| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|         NULL|
|  male|50s|   Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|   2020-02-19|         NULL|
|  male|20s|     Mapo-gu|     overseas inflow|       NULL|             9|        2020-01-26|    2020-01-30|   2020-02-15|         NULL|
|female|20s| Seongbuk-gu|contact with patient| 1

## Data preprocessing

### Fill the nulls in the deceased_date with the released_date.
- You can use <b>coalesce</b> function

In [12]:
from pyspark.sql.functions import coalesce
df = df.withColumn("deceased_date", coalesce(df["deceased_date"], df["released_date"]))
df.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|

### Add a column named no_days which is difference between the deceased_date and the confirmed_date then show the top 5 rows. Print the schema.
- <b> Hint: You need to typecast these columns as date first <b>

In [15]:
from pyspark.sql.functions import datediff, col

df = df.withColumn("no_days", datediff(col("deceased_date"), col("confirmed_date")))
df.show(5)
df.printSchema()

+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|patient_id|   sex|age|country|province|       city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|no_days|
+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|1000000001|  male|50s|  Korea|   Seoul| Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|     13|
|1000000002|  male|30s|  Korea|   Seoul|Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|     32|
|1000000003|  male|50s|  Korea|   Seoul|  Jongno-gu|contact with patient| 2002000001|            17|

In [16]:
df.groupBy("sex").count().show()

+------+-----+
|   sex|count|
+------+-----+
|  NULL| 1122|
|female| 2218|
|  male| 1825|
+------+-----+



### Remove null values of sex column.
### Add a is_male column if male then it should yield true, else (Female) then False

In [17]:
from pyspark.sql import functions as F
df_cleaned = df.dropna(subset=["sex"])
df_final = df_cleaned.withColumn(
    "is_male",
    F.when(F.col("sex") == "male", True).otherwise(False)
)

df_final.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+-------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|no_days|is_male|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+-------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|     13|   true|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|     32|   true|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|co

### Add a is_dead column if patient state is not released then it should yield true, else then False

- Use <b>UDF</b> to perform this task.
- However, UDF is not recommended there is no built in function can do the required operation.
- UDF is slower than built in functions.

In [21]:
df_fast = df.withColumn("is_dead", col("state") != "released")
df_fast.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+-------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|no_days|is_dead|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+-------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|     13|  false|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|     32|  false|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|co

### Change the ages to bins from 10s, 0s, 10s, 20s,.etc to 0,10, 20

In [23]:
df_binned = df.withColumn("age_bin", (F.floor(F.regexp_replace(col("age"), "s", "").cast("int") / 10) * 10).cast("int"))

df_binned.select("age", "age_bin").show()

+---+-------+
|age|age_bin|
+---+-------+
|50s|     50|
|30s|     30|
|50s|     50|
|20s|     20|
|20s|     20|
|50s|     50|
|20s|     20|
|20s|     20|
|30s|     30|
|60s|     60|
|50s|     50|
|20s|     20|
|80s|     80|
|60s|     60|
|70s|     70|
|70s|     70|
|70s|     70|
|20s|     20|
|70s|     70|
|70s|     70|
+---+-------+
only showing top 20 rows


### Change age, and no_days  to be typecasted as Double

In [24]:
df_typed = df.withColumn("age", col("age").cast("double"))
df_typed = df_typed.withColumn("no_days", col("no_days").cast("double"))
df_typed.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: double (nullable = true)
 |-- country: string (nullable = true)
 |-- province: string (nullable = true)
 |-- city: string (nullable = true)
 |-- infection_case: string (nullable = true)
 |-- infected_by: string (nullable = true)
 |-- contact_number: string (nullable = true)
 |-- symptom_onset_date: string (nullable = true)
 |-- confirmed_date: date (nullable = true)
 |-- released_date: date (nullable = true)
 |-- deceased_date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- no_days: double (nullable = true)



### Drop the columns
["patient_id","sex","infected_by","contact_number","released_date","state",
"symptom_onset_date","confirmed_date","deceased_date","country","no_days",
"city","infection_case"]

In [25]:
df_dropped = df.drop("patient_id","sex","infected_by","contact_number","released_date","state",
"symptom","onset_date","confirmed_date","deceased_date","country","no_days",
"city","infection_case")
df_dropped.show()

+---+--------+------------------+
|age|province|symptom_onset_date|
+---+--------+------------------+
|50s|   Seoul|        2020-01-22|
|30s|   Seoul|              NULL|
|50s|   Seoul|              NULL|
|20s|   Seoul|        2020-01-26|
|20s|   Seoul|              NULL|
|50s|   Seoul|              NULL|
|20s|   Seoul|              NULL|
|20s|   Seoul|              NULL|
|30s|   Seoul|              NULL|
|60s|   Seoul|              NULL|
|50s|   Seoul|              NULL|
|20s|   Seoul|              NULL|
|80s|   Seoul|              NULL|
|60s|   Seoul|        2020-02-06|
|70s|   Seoul|        2020-02-11|
|70s|   Seoul|              NULL|
|70s|   Seoul|              NULL|
|20s|   Seoul|              NULL|
|70s|   Seoul|              NULL|
|70s|   Seoul|              NULL|
+---+--------+------------------+
only showing top 20 rows


### Recount the number of nulls now

In [27]:
df_dropped.select([c for c in df_dropped.columns if df_dropped.filter(df_dropped[c].isNull()).count() > 0]).show()

+---+------------------+
|age|symptom_onset_date|
+---+------------------+
|50s|        2020-01-22|
|30s|              NULL|
|50s|              NULL|
|20s|        2020-01-26|
|20s|              NULL|
|50s|              NULL|
|20s|              NULL|
|20s|              NULL|
|30s|              NULL|
|60s|              NULL|
|50s|              NULL|
|20s|              NULL|
|80s|              NULL|
|60s|        2020-02-06|
|70s|        2020-02-11|
|70s|              NULL|
|70s|              NULL|
|20s|              NULL|
|70s|              NULL|
|70s|              NULL|
+---+------------------+
only showing top 20 rows


## Now do the same but using SQL select statement

### From the original Patient DataFrame, Create a temporary view (table).

In [28]:
df.createOrReplaceTempView("patient")

### Use SELECT statement to select all columns from the dataframe and show the output.

In [29]:
df_sql = spark.sql("SELECT * FROM patient")
df_sql.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|no_days|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|     13|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|     32|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 2002000001|         

### *Using SQL commands*, limit the output to only 5 rows

In [30]:
df_sql = spark.sql("SELECT * FROM patient LIMIT 5")
df_sql.show()

+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|patient_id|   sex|age|country|province|       city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|no_days|
+----------+------+---+-------+--------+-----------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+-------+
|1000000001|  male|50s|  Korea|   Seoul| Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|   2020-02-05|released|     13|
|1000000002|  male|30s|  Korea|   Seoul|Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|   2020-03-02|released|     32|
|1000000003|  male|50s|  Korea|   Seoul|  Jongno-gu|contact with patient| 2002000001|            17|

### Select the count of males and females in the dataset

In [31]:
df_sql = spark.sql("SELECT sex, COUNT(*) AS count FROM patient GROUP BY sex")
df_sql.show()

+------+-----+
|   sex|count|
+------+-----+
|  NULL| 1122|
|female| 2218|
|  male| 1825|
+------+-----+



### How many people did survive, and how many didn't?

In [32]:
df_sql = spark.sql("SELECT state, COUNT(*) AS count FROM patient GROUP BY state")
df_sql.show()

+--------+-----+
|   state|count|
+--------+-----+
|isolated| 2158|
|released| 2929|
|deceased|   78|
+--------+-----+



### Now, let's perform some preprocessing using SQL:
1. Convert *age* column to double after removing the 's' at the end -- *hint: check SUBSTRING method*
2. Select only the following columns: `['sex', 'age', 'province', 'state']`
3. Store the result of the query in a new dataframe

In [33]:
df_sql = spark.sql("SELECT CAST(SUBSTRING(age, 1, LENGTH(age) - 1) AS DOUBLE) AS age FROM patient")
df_sql.show()

+----+
| age|
+----+
|50.0|
|30.0|
|50.0|
|20.0|
|20.0|
|50.0|
|20.0|
|20.0|
|30.0|
|60.0|
|50.0|
|20.0|
|80.0|
|60.0|
|70.0|
|70.0|
|70.0|
|20.0|
|70.0|
|70.0|
+----+
only showing top 20 rows


In [34]:
df_sql = spark.sql("SELECT sex, age, province, state FROM patient")
df_sql.show()

+------+---+--------+--------+
|   sex|age|province|   state|
+------+---+--------+--------+
|  male|50s|   Seoul|released|
|  male|30s|   Seoul|released|
|  male|50s|   Seoul|released|
|  male|20s|   Seoul|released|
|female|20s|   Seoul|released|
|female|50s|   Seoul|released|
|  male|20s|   Seoul|released|
|  male|20s|   Seoul|released|
|  male|30s|   Seoul|released|
|female|60s|   Seoul|released|
|female|50s|   Seoul|released|
|  male|20s|   Seoul|released|
|  male|80s|   Seoul|deceased|
|female|60s|   Seoul|released|
|  male|70s|   Seoul|released|
|  male|70s|   Seoul|released|
|  male|70s|   Seoul|released|
|  male|20s|   Seoul|released|
|female|70s|   Seoul|released|
|female|70s|   Seoul|released|
+------+---+--------+--------+
only showing top 20 rows


In [35]:
df_new = df_sql.select("sex", "age", "province", "state")
df_new.show()

+------+---+--------+--------+
|   sex|age|province|   state|
+------+---+--------+--------+
|  male|50s|   Seoul|released|
|  male|30s|   Seoul|released|
|  male|50s|   Seoul|released|
|  male|20s|   Seoul|released|
|female|20s|   Seoul|released|
|female|50s|   Seoul|released|
|  male|20s|   Seoul|released|
|  male|20s|   Seoul|released|
|  male|30s|   Seoul|released|
|female|60s|   Seoul|released|
|female|50s|   Seoul|released|
|  male|20s|   Seoul|released|
|  male|80s|   Seoul|deceased|
|female|60s|   Seoul|released|
|  male|70s|   Seoul|released|
|  male|70s|   Seoul|released|
|  male|70s|   Seoul|released|
|  male|20s|   Seoul|released|
|female|70s|   Seoul|released|
|female|70s|   Seoul|released|
+------+---+--------+--------+
only showing top 20 rows
